In [61]:
import numpy as np
from scipy.optimize import newton

In [62]:
GAMMA = 1.4
M0 = 2.5

L1 = 0.5
L2 = 0.5

theta1_deg = 4.0
y_start = 1.0
y_lip = 0.0

In [63]:
def get_theta(M, beta):
    numerator = 2.0 / np.tan(beta) * (M**2 * np.sin(beta)**2 - 1.0)
    denominator = M**2 * (GAMMA + np.cos(2.0 * beta)) + 2.0
    return np.arctan(numerator / denominator)

def get_beta(M, theta):
    def f(beta):
        return get_theta(M, beta) - theta
    
    beta_0 = np.arcsin(1.0 / M) + theta + np.deg2rad(1.0)
    beta = newton(f, beta_0, maxiter=100, tol=1.0e-12)

    return beta

def normal_shock(Mn1):
    numerator = 1.0 + 0.5 * (GAMMA - 1.0) * Mn1**2
    denominator = GAMMA * Mn1**2 - 0.5 * (GAMMA - 1.0)

    Mn2 = np.sqrt(numerator / denominator)
    pressure_ratio = 1.0 + 2.0 * GAMMA / (GAMMA + 1.0) * (Mn1**2 - 1.0)

    return Mn2, pressure_ratio

def oblique_shock(M1, theta):
    beta = get_beta(M1, theta)

    Mn1 = M1 * np.sin(beta)
    Mn2, pressure_ratio = normal_shock(Mn1)

    M2 = Mn2 / np.sin(beta - theta)

    return beta, M2, pressure_ratio

def line_intersection(p1, angle1, p2, angle2):
    x1, y1 = p1[0], p1[1]
    x2, y2 = p2[0], p2[1]

    c1 = np.cos(angle1)
    s1 = np.sin(angle1)
    c2 = np.cos(angle2)
    s2 = np.sin(angle2)

    det = c1 * (-s2) - s1 * (-c2)

    rhs_x = x2 - x1
    rhs_y = y2 - y1

    s = (rhs_x * (-s2) - rhs_y * (-c2)) / det

    return x1 + s * c1, y1 + s * s1

In [64]:
# First Corner
P1 = (0.0, y_start)

theta1 = np.radians(theta1_deg)
beta1, M1, pressure_ratio1 = oblique_shock(M0, theta1)

x_lip = P1[0] + (P1[1] - y_lip) / np.tan(beta1)
P_lip = (x_lip, y_lip)

# Second Corner
P2 = (L1, y_start - L1 * np.tan(theta1))

beta2 = np.arctan((P2[1] - y_lip) / (x_lip - P2[0])) - theta1
theta2 = get_theta(M1, beta2)
M2 = oblique_shock(M1, theta2)[1]
pressure_ratio2 = oblique_shock(M1, theta2)[2]

# Third Corner
P3 = (L1 + L2, P2[1] - L2 * np.tan(theta1 + theta2))

beta3 = np.arctan((P3[1] - y_lip) / (x_lip - P3[0])) - (theta1 + theta2)
theta3 = get_theta(M2, beta3)
M3 = oblique_shock(M2, theta3)[1]
pressure_ratio3 = oblique_shock(M2, theta3)[2]

# Convex Corner
theta123 = theta1 + theta2 + theta3
beta4, M4, pressure_ratio4 = oblique_shock(M3, theta123)

P_covex = line_intersection(P_lip, beta4 - theta123, P3, -(theta123))

overall_pressure_ratio = pressure_ratio1 * pressure_ratio2 * pressure_ratio3 * pressure_ratio4
stagnation_pressure_ratio = overall_pressure_ratio * (1.0 + 0.5 * (GAMMA - 1.0) * M4**2)**(GAMMA / (GAMMA - 1.0)) / (1.0 + 0.5 * (GAMMA - 1.0) * M0**2)**(GAMMA / (GAMMA - 1.0))

# Output
print(f"P1: {P1[0]:.6f}, {P1[1]:.6f}")
print(f"P2: {P2[0]:.6f}, {P2[1]:.6f}")
print(f"P3: {P3[0]:.6f}, {P3[1]:.6f}")
print(f"P_lip: {P_lip[0]:.6f}, {P_lip[1]:.6f}")
print(f"P_convex: {P_covex[0]:.6f}, {P_covex[1]:.6f}")
print(f"Overall Pressure Ratio: {overall_pressure_ratio:.6f}")
print(f"Stagnation Pressure Ratio: {stagnation_pressure_ratio:.6f}")

P1: 0.000000, 1.000000
P2: 0.500000, 0.965037
P3: 1.000000, 0.891302
P_lip: 1.996172, 0.000000
P_convex: 2.664171, 0.435590
Overall Pressure Ratio: 5.525165
Stagnation Pressure Ratio: 0.942895


In [65]:
L_lower = 2.0

phi_deg_list = [-3.0, -2.3, -1.5, -0.7, 0.0]

x0, y0 = P_lip
n_seg = len(phi_deg_list) - 1
dx = L_lower / n_seg

points = [(float(x0), float(y0))]

x = float(x0)
y = float(y0)

for phi_deg in phi_deg_list[:-1]:
    phi = np.deg2rad(phi_deg)

    x += dx
    y += dx * np.tan(phi)

    points.append((float(x), float(y)))

for i, p in enumerate(points):
    print(f"P_lower_{i}: ({p[0]:.6f}, {p[1]:.6f})")

P_lower_0: (1.996172, 0.000000)
P_lower_1: (2.496172, -0.026204)
P_lower_2: (2.996172, -0.046286)
P_lower_3: (3.496172, -0.059379)
P_lower_4: (3.996172, -0.065488)
